# Fraud Agent — Evaluation (deterministic test set)

**Goal:** Reproducible, presentable evaluation on OOT data. No LLM for data or question generation.

**Data:** Test cases are built **deterministically** from `oot_test_index.csv`:
- Pick users with enough transactions in the test period (≥15 tx).
- For each user: (1) full period, all categories; (2) full period, one category they have; (3) sub-period Dec 21–28 when they have enough tx.

**Metrics:** Same three as e2e — Relevance, Completeness, Readability (1–5), plus Mean. LLM-as-judge with temperature=0.

**Output:** Clean results table and CSV.

## 1. Setup and load OOT

In [2]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OOT_CSV_PATH = Path("/Users/zhumiban/Desktop/agent_bank/bank_docs/models/oot_test_index.csv")
DATE_COL = "transaction_datetime"
USER_COL = "customer_id_number"
FRAUD_COL = "is_fraud"
ID_COL = "transaction_id"
CATEGORY_COL = "merchant_category"

df = pd.read_csv(OOT_CSV_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
test_start = df[DATE_COL].min()
test_end = df[DATE_COL].max()
print(f"OOT: {test_start.date()} to {test_end.date()}, rows={len(df)}, users={df[USER_COL].nunique()}")

OOT: 2020-12-07 to 2020-12-31, rows=111144, users=889


## 2. Build test cases (deterministic from data)

In [3]:
# Reproducible: users with >= MIN_TX in test period, sorted by user_id
MIN_TX_FULL = 15
MIN_TX_SUBPERIOD = 5
TARGET_N_CASES = 50
DEC21, DEC28 = "2020-12-21", "2020-12-28"

user_tx = df.groupby(df[USER_COL].astype(str), sort=True).size()
candidates = user_tx[user_tx >= MIN_TX_FULL].index.tolist()
eval_users = candidates[:25]

tests = []
for uid in eval_users:
    sub = df[df[USER_COL].astype(str) == uid]
    # (1) Full period, all categories
    tests.append({
        "case_id": len(tests) + 1,
        "label": f"Full period, all categories",
        "current_user_id": uid,
        "question": "Is there any indication of suspicious activity in my transactions between 2020-12-07 and 2020-12-31?",
        "time_range": "2020-12-07 to 2020-12-31",
        "category": "all",
    })
    # (2) Full period, one category (first category with >=3 tx)
    cat_counts = sub[CATEGORY_COL].value_counts()
    for cat in cat_counts.index:
        if cat_counts[cat] >= 3:
            break
    else:
        cat = sub[CATEGORY_COL].iloc[0]
    tests.append({
        "case_id": len(tests) + 1,
        "label": f"Full period, {cat}",
        "current_user_id": uid,
        "question": f"Can you tell me if any {cat} transactions between 2020-12-07 and 2020-12-31 were fraudulent?",
        "time_range": "2020-12-07 to 2020-12-31",
        "category": str(cat),
    })
    # (3) Dec 21-28, all categories (only if user has enough tx)
    sub_dec = sub[(sub[DATE_COL] >= DEC21) & (sub[DATE_COL] <= DEC28)]
    if len(sub_dec) >= MIN_TX_SUBPERIOD:
        tests.append({
            "case_id": len(tests) + 1,
            "label": "Dec 21-28, all categories",
            "current_user_id": uid,
            "question": "Between December 21 and December 28, 2020, were there any suspicious or fraudulent transactions?",
            "time_range": "2020-12-21 to 2020-12-28",
            "category": "all",
        })

tests = tests[:TARGET_N_CASES]
import random
random.seed(42)
random.shuffle(tests)
for i, t in enumerate(tests):
    t["case_id"] = i + 1

print(f"Built {len(tests)} test cases (target {TARGET_N_CASES}) from {len(eval_users)} users (shuffled, seed=42).")
for t in tests[:5]:
    print(f"  {t['case_id']}: {t['label']} (user={str(t['current_user_id'])[:18]}...)")
print(f"  ... and {len(tests) - 5} more.")

Built 50 test cases (target 50) from 25 users (shuffled, seed=42).
  1: Full period, grocery_pos (user=180046000000000.0...)
  2: Dec 21-28, all categories (user=180043000000000.0...)
  3: Full period, gas_transport (user=180040000000000.0...)
  4: Dec 21-28, all categories (user=180018000000000.0...)
  5: Full period, shopping_pos (user=180014000000000.0...)
  ... and 45 more.


## 3. Run fraud agent

In [4]:
import time
from src.graph.fraud_agent.pipeline import run_fraud_agent

SLEEP_AGENT = 4.0
MAX_RETRIES_429 = 2
RETRY_SLEEPS = [20, 45]
# Warmup so first API burst is not on a counted case
run_fraud_agent(question=tests[0]["question"], current_user_id=tests[0]["current_user_id"])
time.sleep(5)
results = []
for t in tests:
    out = None
    for attempt in range(MAX_RETRIES_429 + 1):
        out = run_fraud_agent(question=t["question"], current_user_id=t["current_user_id"])
        err = out.get("error") or ""
        if "429" not in str(err):
            break
        if attempt < MAX_RETRIES_429:
            wait = RETRY_SLEEPS[attempt]
            print(f"Case {t['case_id']}: 429 rate limit, retry in {wait}s...")
            time.sleep(wait)
    results.append({
        **t,
        "analysis": out.get("analysis") or "",
        "error": out.get("error"),
        "risk_scores": out.get("risk_scores"),
    })
    time.sleep(SLEEP_AGENT)
print(f"Ran {len(results)} agent calls.")

Case 7: 429 rate limit, retry in 20s...
Case 9: 429 rate limit, retry in 20s...
Case 17: 429 rate limit, retry in 20s...
Case 21: 429 rate limit, retry in 20s...
Case 25: 429 rate limit, retry in 20s...
Case 27: 429 rate limit, retry in 20s...
Case 28: 429 rate limit, retry in 20s...
Case 30: 429 rate limit, retry in 20s...
Case 35: 429 rate limit, retry in 20s...
Case 39: 429 rate limit, retry in 20s...
Case 40: 429 rate limit, retry in 20s...
Case 41: 429 rate limit, retry in 20s...
Case 46: 429 rate limit, retry in 20s...
Case 47: 429 rate limit, retry in 20s...
Case 49: 429 rate limit, retry in 20s...
Ran 50 agent calls.


## 4. Score with Relevance, Completeness, Readability

In [5]:
import re
import time
from src.models.llm import get_llm

EVAL_RUBRIC = """You are an expert evaluator assessing the quality of an AI agent's response in a banking fraud analysis setting. Your task is to score the response on three dimensions, each on an integer scale from 1 to 5.

**Scoring dimensions:**

1. **Relevance** (1-5): Does the response directly address the user's question? Consider alignment with the requested time range, scope (e.g., category or account), and intent (fraud check / risk analysis). 1 = off-topic or wrong context; 5 = fully on-topic and contextually correct.

2. **Completeness** (1-5): Does the response include all necessary information? Consider: clear conclusion, presence of a fraud/high-risk transaction list or explicit statement of none, and any relevant risk summary. 1 = missing critical elements or empty; 5 = complete and self-contained.

3. **Readability** (1-5): Is the response human-readable and well-structured? Consider: clear sections, coherent prose, no broken placeholders, appropriate tone. 1 = unreadable or severely broken; 5 = clear, professional, and easy to follow.

**Input:** You will be given the user question and the agent's full response.

**Output:** Reply with exactly three lines in this format (no other text before or after):
RELEVANCE: <integer 1-5>
COMPLETENESS: <integer 1-5>
READABILITY: <integer 1-5>"""

def parse_scores(raw: str):
    raw = raw.strip().upper()
    r = re.search(r"RELEVANCE:\s*(\d)", raw)
    c = re.search(r"COMPLETENESS:\s*(\d)", raw)
    rd = re.search(r"READABILITY:\s*(\d)", raw)
    if r and c and rd:
        return (int(r.group(1)), int(c.group(1)), int(rd.group(1)))
    return (None, None, None)

def score_one(user_question: str, agent_response: str, llm_client):
    content = f"User question:\n{user_question}\n\nAgent response:\n{agent_response or '(empty)'}"
    raw = llm_client.chat(system_prompt=EVAL_RUBRIC, user_prompt=content).strip()
    return parse_scores(raw)

SLEEP_EVAL = 0.5
llm_eval = get_llm(role="generic", temperature=0.0)
for r in results:
    rel, comp, read = score_one(r["question"], r.get("analysis") or "", llm_eval)
    r["Relevance"] = rel
    r["Completeness"] = comp
    r["Readability"] = read
    r["Mean"] = round((rel + comp + read) / 3.0, 2) if (rel is not None and comp is not None and read is not None) else None
    time.sleep(SLEEP_EVAL)
print("Scoring done.")

Scoring done.


## 5. Results table and save

In [8]:
eval_df = pd.DataFrame([{
    "Case": r["case_id"],
    "User ID": r["current_user_id"],
    "Time range": r["time_range"],
    "Category": r["category"],
    "Question": r["question"],
    "Relevance": r.get("Relevance"),
    "Completeness": r.get("Completeness"),
    "Readability": r.get("Readability"),
    "Mean": r.get("Mean"),
    "Error": "429 Rate limit" if (r.get("error") and "429" in str(r.get("error"))) else (str(r.get("error"))[:60] if r.get("error") else ""),
    "Response preview": (r.get("analysis") or ""),
} for r in results])

display(eval_df)
means = eval_df[["Relevance", "Completeness", "Readability", "Mean"]].mean()
print("\nOverall means:")
print(means.to_string())

# Show full Q&A for low-score cases (Mean < 4 or Completeness < 3) to see why
low = [r for r in results if (r.get("Mean") or 5) < 4 or (r.get("Completeness") or 5) < 3]
if low:
    print("\n--- Low-score cases (full question + agent response) ---")
    for r in low:
        print(f"\nCase {r['case_id']} (R={r.get('Relevance')} C={r.get('Completeness')} Read={r.get('Readability')} Mean={r.get('Mean')})")
        print(f"Q: {r['question']}")
        print(f"A: {r.get('analysis') or '(empty)'}")
        if r.get("error"):
            print(f"Error: {r.get('error')}")

out_path = Path.cwd() / "fraud_agent_eval_results.csv"
eval_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"\nSaved: {out_path}")

,Case,User ID,Time range,Category,Question,Relevance,Completeness,Readability,Mean,Error,Response preview
0,1,180046000000000.0,2020-12-07 to 2020-12-31,grocery_pos,Can you tell me if any grocery_pos transaction...,5,5,5,5.00,,Risk Analysis Report\n\n1. Decision: Approve\n...
1,2,180043000000000.0,2020-12-21 to 2020-12-28,all,"Between December 21 and December 28, 2020, wer...",5,5,5,5.00,,Risk Analysis Report: Transactions from Decemb...
2,3,180040000000000.0,2020-12-07 to 2020-12-31,gas_transport,Can you tell me if any gas_transport transacti...,5,5,5,5.00,,Risk Analysis Report for Gas_Transport Transac...
3,4,180018000000000.0,2020-12-21 to 2020-12-28,all,"Between December 21 and December 28, 2020, wer...",5,5,5,5.00,,Risk Analysis Report for Transactions (Decembe...
4,5,180014000000000.0,2020-12-07 to 2020-12-31,shopping_pos,Can you tell me if any shopping_pos transactio...,5,5,5,5.00,,Risk Analysis Report\n\n1. Decision: Approve\n...
5,6,180067000000000.0,2020-12-07 to 2020-12-31,all,Is there any indication of suspicious activity...,5,5,5,5.00,,Risk Analysis Report\n\n1. Decision: Approve\n...
6,7,180046000000000.0,2020-12-21 to 2020-12-28,all,"Between December 21 and December 28, 2020, wer...",5,5,5,5.00,,Risk Analysis Report\n\n1. Decision: Approve\n...
7,8,180018000000000.0,2020-12-07 to 2020-12-31,all,Is there any indication of suspicious activity...,5,5,5,5.00,,Risk Analysis Report\n\n1. Decision: Approve\n...
8,9,180047000000000.0,2020-12-21 to 2020-12-28,all,"Between December 21 and December 28, 2020, wer...",5,5,5,5.00,,"Risk Analysis Report: December 21–28, 2020\n\n..."
9,10,180036000000000.0,2020-12-07 to 2020-12-31,gas_transport,Can you tell me if any gas_transport transacti...,5,5,5,5.00,,Risk Analysis Report: Gas Transport Transactio...



Overall means:
Relevance       4.9
Completeness    4.8
Readability     5.0
Mean            4.9

--- Low-score cases (full question + agent response) ---

Case 15 (R=4 C=2 Read=5 Mean=3.67)
Q: Is there any indication of suspicious activity in my transactions between 2020-12-07 and 2020-12-31?
A: I'm unable to access your transaction data right now due to a temporary system limitation. Therefore, I cannot analyze your activity for suspicious transactions between 2020-12-07 and 2020-12-31 at this moment.

Please try again in a few minutes, and I'll be able to provide a detailed risk analysis report for your requested period.

Case 19 (R=4 C=2 Read=5 Mean=3.67)
Q: Between December 21 and December 28, 2020, were there any suspicious or fraudulent transactions?
A: Sorry, I was unable to retrieve the transaction data for December 21 to December 28, 2020, due to a temporary system error. Please try again in a few moments, and I will re-run the analysis for suspicious or fraudulent transaction